# 07 · NLP aplicado al riesgo de crédito

| | |
|---|---|
| **Fase** | F3: Analizar y modelar |
| **IDs del enunciado que cubre** | **IA3.1** (NLP en datos financieros: análisis de texto, *Sentiment Analysis*, BERT y GPT), **IA3.E1** (modelo de NLP para la predicción de riesgo), **R2** (técnicas de análisis de datos, IA y NLP) |
| **Entregables** | este notebook + `Back/models/modelo_nlp_nb07.joblib` + `Back/models/resultados_nlp.json` |

**Contenido:**
1. Análisis de texto clásico: limpieza (minúsculas, tildes, *stopwords*), tokenización, frecuencias, n-gramas y palabras asociadas al impago.
2. Línea base NLP: `TfidfVectorizer + LogisticRegression` solo con el texto.
3. *Sentiment Analysis* con BERT multilingüe (`nlptown/bert-base-multilingual-uncased-sentiment`).
4. Embeddings con BERT (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) vs **GPT-2** en español (`datificate/gpt2-small-spanish`, con respaldo `gpt2`): promedio del último *hidden state*.
5. Modelo combinado (tabular + sentimiento + embeddings reducidos) y su **ΔAUC** frente al modelo tabular.

> Todo es local y gratuito (Hugging Face), **sin APIs de pago**. Para que corra rápido en CPU, BERT y GPT se aplican a una **muestra estratificada de 3.000 solicitudes** y sus resultados se cachean en `Back/notebooks/cache/`.
> El modelo NLP que sirve la API (`Back/models/modelo_nlp.joblib`) lo genera el script de entrenamiento del back; este notebook guarda su versión como `modelo_nlp_nb07.joblib`.

**Prerrequisitos:** `scikit-learn`, `unidecode` (opcional), y para BERT/GPT: `pip install torch transformers sentence-transformers`. La primera ejecución descarga los modelos (~1 GB); luego se usan desde la caché local.

## 0. Configuración

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.options.display.float_format = "{:,.4f}".format
pd.options.display.max_columns = 60
plt.rcParams["figure.dpi"] = 100

def encontrar_raiz():
    """Sube desde el directorio actual hasta encontrar db/credit_risk.csv (sin rutas absolutas)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "db" / "credit_risk.csv").exists():
            return carpeta
    raise FileNotFoundError("No se encontró db/credit_risk.csv. Ejecute el notebook desde Back/notebooks.")

RAIZ = encontrar_raiz()
DATA_PATH = RAIZ / "db" / "credit_risk.csv"
MODELS_DIR = RAIZ / "Back" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_PATH.relative_to(RAIZ), "| artefactos:", MODELS_DIR.relative_to(RAIZ))

In [ ]:
TARGET = "default"
TEXTO = "texto_solicitud"
SENSIBLES = ["sexo"]   # solo para auditar equidad, NUNCA como predictor
NUM_BASE = ["edad", "antiguedad_laboral_anios", "personas_a_cargo", "ingreso_mensual", "monto_credito",
            "plazo_meses", "tasa_interes_anual", "num_creditos_activos", "moras_ultimos_12m",
            "relacion_deuda_ingreso", "puntaje_buro", "uso_cupo_tarjetas"]
CAT_COLS = ["estado_civil", "nivel_educativo", "situacion_laboral", "tipo_vivienda", "proposito"]
NUEVAS = ["cuota_estimada", "ratio_cuota_ingreso", "ratio_monto_ingreso", "tiene_moras_previas", "longitud_texto"]
NUM_FEATS = NUM_BASE + NUEVAS
FEATURES = NUM_FEATS + CAT_COLS

def limpiar(df_raw, verbose=True):
    """Duplicados (exactos y por id_cliente), valores inválidos -> NaN y texto vacío.
    La imputación (mediana/moda) y el capping p1-p99 se hacen en `preparar`, con estadísticos de train."""
    df = df_raw.copy()
    n0 = len(df)
    df = df.drop_duplicates()
    n1 = len(df)
    df = df.drop_duplicates(subset="id_cliente", keep="first")
    n2 = len(df)
    df = df[df[TARGET].isin([0, 1])].copy()
    df[TARGET] = df[TARGET].astype(int)
    df.loc[~df["edad"].between(18, 100), "edad"] = np.nan          # edades imposibles (1, 999...)
    for c in ["ingreso_mensual", "monto_credito"]:
        df.loc[df[c] <= 0, c] = np.nan
    for c in CAT_COLS + ["sexo"]:
        df[c] = df[c].where(df[c].isna(), df[c].astype(str).str.strip())
    df[TEXTO] = df[TEXTO].fillna("").astype(str)
    if verbose:
        print(f"Filas originales: {n0:,} | duplicados exactos: {n0 - n1} | duplicados por id_cliente: {n1 - n2} | filas finales: {len(df):,}")
    return df.reset_index(drop=True)

def agregar_variables(df):
    """Variables de negocio del paso 3: cuota (amortización francesa), ratios, moras previas y longitud del texto."""
    df = df.copy()
    r = df["tasa_interes_anual"] / 100 / 12
    n = df["plazo_meses"]
    df["cuota_estimada"] = df["monto_credito"] * r / (1 - (1 + r) ** (-n))
    df["ratio_cuota_ingreso"] = df["cuota_estimada"] / df["ingreso_mensual"]
    df["ratio_monto_ingreso"] = df["monto_credito"] / df["ingreso_mensual"]
    df["tiene_moras_previas"] = (df["moras_ultimos_12m"] > 0).astype(int)
    df["longitud_texto"] = df[TEXTO].str.len()
    return df

def preparar(df_limpio, test_size=0.2):
    """Split estratificado ANTES de imputar/capar: medianas, modas y percentiles salen solo de train (sin leakage)."""
    train, test = train_test_split(df_limpio, test_size=test_size, stratify=df_limpio[TARGET],
                                   random_state=RANDOM_STATE)
    train, test = train.copy(), test.copy()
    medianas = train[NUM_BASE].median()
    modas = {c: train[c].mode().iloc[0] for c in CAT_COLS}
    p01, p99 = train[NUM_BASE].quantile(0.01), train[NUM_BASE].quantile(0.99)
    preparar.stats = {"medianas": medianas, "modas": modas, "p01": p01, "p99": p99}
    salida = []
    for d in (train, test):
        d[NUM_BASE] = d[NUM_BASE].fillna(medianas).clip(lower=p01, upper=p99, axis=1)
        for c in CAT_COLS:
            d[c] = d[c].fillna(modas[c])
        salida.append(agregar_variables(d).reset_index(drop=True))
    return salida[0], salida[1]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor(extra_num=()):
    """Mismo preprocesador del paso 3: numéricas (mediana + StandardScaler) y categóricas (moda + One-Hot)."""
    num = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_FEATS + list(extra_num)), ("cat", cat, CAT_COLS)])

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score,
                             recall_score, f1_score, brier_score_loss, roc_curve, precision_recall_curve,
                             confusion_matrix)

def ks_stat(y, p):
    """Estadístico KS: máxima separación entre las distribuciones acumuladas de buenos y malos."""
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))

def metricas(y, p, umbral=0.5):
    pred = (np.asarray(p) >= umbral).astype(int)
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred), "ks": ks_stat(y, p),
            "brier": brier_score_loss(y, p)}

## 1. Carga y limpieza inline

Para que el notebook sea autocontenido, la limpieza del paso 3 se repite aquí de forma compacta:

- **Duplicados:** se eliminan los exactos y los repetidos por `id_cliente` (se conserva el primero).
- **Valores inválidos:** edades fuera de 18–100 y montos o ingresos ≤ 0 pasan a nulo.
- **Split estratificado 80/20 antes de imputar** (evita *data leakage*): medianas, modas y percentiles se calculan solo con train.
- **Imputación:** mediana en las numéricas y moda en las categóricas.
- **Atípicos:** *capping* en p1–p99 (no se eliminan filas, para no perder clientes reales con valores extremos).
- **Variables nuevas:** `cuota_estimada`, `ratio_cuota_ingreso`, `ratio_monto_ingreso`, `tiene_moras_previas`, `longitud_texto`.
- `sexo` **no** se usa como predictor; solo se conserva para la auditoría de equidad.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Forma original:", df_raw.shape)
print("Nulos por columna (solo las que tienen):")
print(df_raw.isna().sum()[lambda s: s > 0].to_string())
df = limpiar(df_raw)
train, test = preparar(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Train: {X_train.shape} | tasa de default {y_train.mean():.3f}")
print(f"Test:  {X_test.shape} | tasa de default {y_test.mean():.3f}")
print("Nulos restantes en features (train/test):", int(train[FEATURES].isna().sum().sum()), int(test[FEATURES].isna().sum().sum()))

## 2. Análisis de texto clásico

Limpieza: minúsculas, eliminación de tildes, de números y de signos, y de *stopwords* en español (lista incluida en el notebook para no depender de descargas).

In [ ]:
import re, unicodedata
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

STOPWORDS_ES = set("""a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante e el ella
ellas ellos en entre era eran es esa esas ese eso esos esta estaba estado estan estar este esto estos estoy fue fueron
ha habia han hasta hay he la las le les lo los mas me mi mis mucho muy nada ni no nos nosotros o os otra otro para pero
poco por porque que quien se sea ser si sido sin sobre son su sus tambien tanto te tengo tiene tienen todo todos tu tus
un una uno unos y ya yo""".split())

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r"[^a-zñ\s]", " ", texto)
    return " ".join(w for w in texto.split() if w not in STOPWORDS_ES and len(w) > 2)

df["texto_limpio"] = df[TEXTO].map(normalizar)
train["texto_limpio"] = train[TEXTO].map(normalizar)
test["texto_limpio"] = test[TEXTO].map(normalizar)
print("Textos vacíos (nulos imputados con ''):", int((df[TEXTO].str.len() == 0).sum()))
df[[TEXTO, "texto_limpio"]].head(3)

In [ ]:
tokens = Counter(w for t in df["texto_limpio"] for w in t.split())
print("Vocabulario:", len(tokens), "| tokens totales:", sum(tokens.values()))
print("Longitud media (palabras):", round(df["texto_limpio"].str.split().str.len().mean(), 1))
cv_bi = CountVectorizer(ngram_range=(2, 2), min_df=5)
B = cv_bi.fit_transform(df["texto_limpio"])
bigramas = pd.Series(np.asarray(B.sum(axis=0)).ravel(), index=cv_bi.get_feature_names_out()).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.Series(dict(tokens.most_common(15))).sort_values().plot.barh(ax=axes[0], color="tab:blue")
axes[0].set(title="15 palabras más frecuentes", xlabel="Frecuencia")
bigramas.head(15).sort_values().plot.barh(ax=axes[1], color="tab:orange")
axes[1].set(title="15 bigramas más frecuentes", xlabel="Frecuencia")
plt.tight_layout(); plt.show()

### 2.1 Palabras más asociadas a `default = 1` vs `default = 0`

Se usa el **log-odds ratio** suavizado: log[(f₁ + 1) / (N₁ + V)] − log[(f₀ + 1) / (N₀ + V)], donde f es la frecuencia de la palabra en cada clase.

In [ ]:
cv_uni = CountVectorizer(ngram_range=(1, 2), min_df=10)
M = cv_uni.fit_transform(df["texto_limpio"])
vocab = cv_uni.get_feature_names_out()
f1 = np.asarray(M[df[TARGET].values == 1].sum(axis=0)).ravel()
f0 = np.asarray(M[df[TARGET].values == 0].sum(axis=0)).ravel()
V = len(vocab)
log_odds = pd.Series(np.log((f1 + 1) / (f1.sum() + V)) - np.log((f0 + 1) / (f0.sum() + V)), index=vocab).sort_values()
top_asoc = pd.concat([log_odds.head(12), log_odds.tail(12)])
plt.figure(figsize=(8, 7))
top_asoc.plot.barh(color=["tab:green" if v < 0 else "tab:red" for v in top_asoc])
plt.axvline(0, color="k", lw=0.8)
plt.title("Términos asociados a impago (rojo) y a buen pago (verde)"); plt.xlabel("Log-odds ratio")
plt.tight_layout(); plt.show()

**Interpretación:** los términos en rojo (por ejemplo, relacionados con atrasos, pérdida de empleo o urgencia) aparecen proporcionalmente más en solicitudes que terminan en impago; los verdes (cumplimiento, estabilidad laboral) en las de buen pago. Esto justifica usar el texto como fuente de información adicional.

## 3. Línea base NLP: TF-IDF + Regresión logística (solo texto, dataset completo)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

modelo_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode", stop_words=sorted(STOPWORDS_ES),
                              ngram_range=(1, 2), min_df=3, max_features=20000, sublinear_tf=True)),
    ("modelo", LogisticRegression(C=2.0, class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE))])
t0 = time.time()
modelo_tfidf.fit(train[TEXTO], y_train)
p_tfidf = modelo_tfidf.predict_proba(test[TEXTO])[:, 1]
met_tfidf = metricas(y_test, p_tfidf)
print(f"TF-IDF + LR (solo texto) en test -> AUC = {met_tfidf['roc_auc']:.4f} | {time.time() - t0:.1f} s")
pd.Series(met_tfidf).round(4).to_frame("TF-IDF + LR")

In [ ]:
import joblib
ruta_nlp = MODELS_DIR / "modelo_nlp_nb07.joblib"
joblib.dump({"pipeline": modelo_tfidf, "metricas_test": met_tfidf, "version": "1.0.0-nb07"}, ruta_nlp)
print("Modelo NLP liviano guardado en", ruta_nlp.relative_to(RAIZ))
print("Ejemplo:", modelo_tfidf.predict_proba(["Me atrase en pagos por perdida de empleo y necesito consolidar deudas"])[:, 1])

## 4. Muestra estratificada para los modelos de lenguaje (BERT y GPT)

In [ ]:
N_MUESTRA = 3000
muestra, _ = train_test_split(df, train_size=N_MUESTRA, stratify=df[TARGET], random_state=RANDOM_STATE)
muestra = muestra.reset_index(drop=True)
train_m, test_m = preparar(muestra)
print("Muestra:", muestra.shape, "| train_m:", train_m.shape, "| test_m:", test_m.shape,
      "| tasa default:", round(muestra[TARGET].mean(), 3))

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
    torch.manual_seed(RANDOM_STATE)
    TRANSFORMERS_OK = True
    print("torch", torch.__version__, "| transformers OK")
except Exception as e:
    TRANSFORMERS_OK = False
    print("transformers/torch no disponible:", repr(e)[:200])
    print("Instalar con: pip install torch transformers")
try:
    from sentence_transformers import SentenceTransformer
    ST_OK = True
except Exception as e:
    ST_OK = False
    print("sentence-transformers no disponible. Instalar con: pip install sentence-transformers")

SENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GPT_MODELS = ["datificate/gpt2-small-spanish", "gpt2"]
CACHE = RAIZ / "Back" / "notebooks" / "cache" / "nlp_features_muestra.parquet"

def cargar_hf(clase, nombre, **kw):
    """Primero intenta la caché local (sin red); si no está, descarga del Hub de Hugging Face."""
    try:
        return clase.from_pretrained(nombre, local_files_only=True, **kw)
    except Exception:
        return clase.from_pretrained(nombre, **kw)

textos_m = [t if t.strip() else "sin texto" for t in muestra[TEXTO]]

## 5. *Sentiment Analysis* con BERT

El modelo `nlptown/bert-base-multilingual-uncased-sentiment` clasifica cada texto en 1–5 estrellas. Se usa el **valor esperado de estrellas** Σ k·P(k) como puntaje continuo de sentimiento (1 = muy negativo, 5 = muy positivo).

In [ ]:
def sentimiento_bert(textos, batch=32):
    tok = cargar_hf(AutoTokenizer, SENT_MODEL)
    mod = cargar_hf(AutoModelForSequenceClassification, SENT_MODEL).eval()
    estrellas, salida = np.arange(1, 6), []
    with torch.no_grad():
        for i in range(0, len(textos), batch):
            enc = tok(textos[i:i + batch], padding=True, truncation=True, max_length=96, return_tensors="pt")
            salida.append(torch.softmax(mod(**enc).logits, dim=-1).numpy() @ estrellas)
    return np.concatenate(salida)

def embeddings_bert(textos):
    try:
        st = SentenceTransformer(EMB_MODEL, local_files_only=True)
    except Exception:
        st = SentenceTransformer(EMB_MODEL)
    return st.encode(textos, batch_size=64, show_progress_bar=False, normalize_embeddings=True)

def embeddings_gpt(textos, batch=32):
    for nombre in GPT_MODELS:
        try:
            tok, mod = cargar_hf(AutoTokenizer, nombre), cargar_hf(AutoModel, nombre).eval()
            break
        except Exception as e:
            print(f"No se pudo cargar {nombre}: {type(e).__name__}")
    tok.pad_token = tok.pad_token or tok.eos_token
    salida = []
    with torch.no_grad():
        for i in range(0, len(textos), batch):
            enc = tok(textos[i:i + batch], padding=True, truncation=True, max_length=96, return_tensors="pt")
            h = mod(**enc).last_hidden_state                       # (batch, tokens, 768)
            m = enc["attention_mask"].unsqueeze(-1).float()
            salida.append(((h * m).sum(1) / m.sum(1).clamp(min=1)).numpy())   # promedio del último hidden state
    print("GPT usado:", nombre)
    return np.vstack(salida), nombre

NLP_OK = TRANSFORMERS_OK and ST_OK
if CACHE.exists() and pd.read_parquet(CACHE, columns=["id_cliente"])["id_cliente"].tolist() == muestra["id_cliente"].tolist():
    feats = pd.read_parquet(CACHE)
    print("Inferencias BERT/GPT leídas de la caché:", CACHE.relative_to(RAIZ))
elif NLP_OK:
    t0 = time.time(); sent = sentimiento_bert(textos_m); t_sent = time.time() - t0
    t0 = time.time(); E_bert = embeddings_bert(textos_m); t_bert = time.time() - t0
    t0 = time.time(); E_gpt, gpt_usado = embeddings_gpt(textos_m); t_gpt = time.time() - t0
    print(f"Tiempos (s) -> sentimiento: {t_sent:.0f} | BERT emb: {t_bert:.0f} | GPT emb: {t_gpt:.0f}")
    feats = pd.concat([pd.DataFrame({"id_cliente": muestra["id_cliente"], "sentimiento": sent}),
                       pd.DataFrame(E_bert, columns=[f"bert_{i}" for i in range(E_bert.shape[1])]),
                       pd.DataFrame(E_gpt, columns=[f"gpt_{i}" for i in range(E_gpt.shape[1])])], axis=1)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    feats.to_parquet(CACHE, index=False)
    print("Caché guardada en", CACHE.relative_to(RAIZ))
else:
    feats = None
    print("Se omiten BERT y GPT: instale con  pip install torch transformers sentence-transformers")

In [ ]:
if feats is not None:
    muestra["sentimiento"] = feats["sentimiento"].values
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    muestra.boxplot(column="sentimiento", by=TARGET, ax=axes[0])
    axes[0].set(title="Sentimiento BERT (1-5 estrellas) por default", xlabel="default", ylabel="Estrellas esperadas")
    muestra.groupby(muestra["sentimiento"].round())[TARGET].mean().plot.bar(ax=axes[1], rot=0, color="tab:purple")
    axes[1].set(title="Tasa de impago por nivel de sentimiento", xlabel="Estrellas (redondeado)", ylabel="Tasa de impago")
    plt.suptitle(""); plt.tight_layout(); plt.show()
    print("Correlación (Spearman) sentimiento vs default:", round(muestra[["sentimiento", TARGET]].corr("spearman").iloc[0, 1], 4))
    print("AUC del sentimiento solo (menor sentimiento = más riesgo):", round(roc_auc_score(muestra[TARGET], -muestra["sentimiento"]), 4))

**Interpretación:** si los clientes que impagan escriben con un tono más negativo (menos estrellas), el sentimiento tendrá AUC > 0.5 y correlación negativa con `default`. El modelo de sentimiento no fue entrenado para riesgo de crédito, por lo que su señal suele ser débil pero complementaria.

## 6. Embeddings BERT vs GPT como entrada de un clasificador

Cada texto se convierte en un vector (384 dimensiones con MiniLM, 768 con GPT-2) y se entrena una regresión logística sobre la parte de entrenamiento de la muestra. Se compara contra TF-IDF entrenado con la misma muestra.

In [ ]:
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.decomposition import PCA

resultados_txt = []
idx_tr = muestra["id_cliente"].isin(train_m["id_cliente"]).values
idx_te = ~idx_tr
y_m = muestra[TARGET].values

def evaluar_texto(nombre, Xtr_, Xte_, modelo):
    modelo.fit(Xtr_, y_m[idx_tr])
    p = modelo.predict_proba(Xte_)[:, 1]
    resultados_txt.append({"modelo": nombre, **metricas(y_m[idx_te], p)})
    return p

tfidf_m = Pipeline([("tfidf", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                    ("modelo", LogisticRegression(C=2.0, class_weight="balanced", max_iter=3000))])
evaluar_texto("TF-IDF + LR (muestra)", muestra.loc[idx_tr, TEXTO], muestra.loc[idx_te, TEXTO], tfidf_m)

if feats is not None:
    E_bert = feats.filter(like="bert_").values
    E_gpt = feats.filter(like="gpt_").values
    for nombre, E in [("Embeddings BERT (MiniLM) + LR", E_bert), ("Embeddings GPT-2 + LR", E_gpt)]:
        evaluar_texto(nombre, E[idx_tr], E[idx_te],
                      Pipeline([("esc", _SS()), ("modelo", LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000))]))
    evaluar_texto("Sentimiento BERT + LR", muestra.loc[idx_tr, ["sentimiento"]], muestra.loc[idx_te, ["sentimiento"]],
                  LogisticRegression(class_weight="balanced"))
tabla_txt = pd.DataFrame(resultados_txt).set_index("modelo").sort_values("roc_auc", ascending=False)
tabla_txt.round(4)

**Interpretación:** BERT (entrenado para similitud semántica multilingüe) suele producir mejores vectores de oración que GPT-2, cuyo objetivo es predecir la siguiente palabra; TF-IDF es un rival fuerte cuando el vocabulario de riesgo es explícito. *Opcional:* GPT también podría usarse en modo *zero-shot* con un *prompt*, pero un GPT-2 pequeño no sigue instrucciones de forma fiable, por lo que no se usa para decidir.

## 7. Modelo combinado: tabular + texto (IA3.E1) y ΔAUC

**7.1 Dataset completo:** se agrega como variable la probabilidad del modelo TF-IDF calculada *out-of-fold* en train (para no filtrar la etiqueta) y la del modelo ajustado en todo train para test.

**7.2 Muestra con BERT:** tabular + sentimiento + 10 componentes PCA de los embeddings BERT (PCA ajustado solo con train).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def modelo_tab(extra=()):
    return Pipeline([("prep", build_preprocessor(extra_num=extra)),
                     ("modelo", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE))])

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train["p_texto"] = cross_val_predict(modelo_tfidf, train[TEXTO], y_train, cv=cv5, method="predict_proba")[:, 1]
test["p_texto"] = p_tfidf

combinados = []
for nombre, extra in [("Tabular", ()), ("Tabular + prob. TF-IDF", ("p_texto",))]:
    m = modelo_tab(extra).fit(train[FEATURES + list(extra)], y_train)
    combinados.append({"datos": "Completo", "modelo": nombre,
                       **metricas(y_test, m.predict_proba(test[FEATURES + list(extra)])[:, 1])})

if feats is not None:
    train_m = train_m.merge(feats[["id_cliente", "sentimiento"]], on="id_cliente")
    test_m = test_m.merge(feats[["id_cliente", "sentimiento"]], on="id_cliente")
    emb = feats.set_index("id_cliente").filter(like="bert_")
    pca = PCA(n_components=10, random_state=RANDOM_STATE).fit(emb.loc[train_m["id_cliente"]].values)
    cols_pca = [f"bert_pca_{i}" for i in range(10)]
    train_m[cols_pca] = pca.transform(emb.loc[train_m["id_cliente"]].values)
    test_m[cols_pca] = pca.transform(emb.loc[test_m["id_cliente"]].values)
    for nombre, extra in [("Tabular", ()), ("Tabular + sentimiento", ("sentimiento",)),
                          ("Tabular + sentimiento + BERT PCA", ("sentimiento", *cols_pca))]:
        m = modelo_tab(extra).fit(train_m[FEATURES + list(extra)], train_m[TARGET])
        combinados.append({"datos": f"Muestra {N_MUESTRA}", "modelo": nombre,
                           **metricas(test_m[TARGET], m.predict_proba(test_m[FEATURES + list(extra)])[:, 1])})

tabla_comb = pd.DataFrame(combinados)
base = tabla_comb[tabla_comb["modelo"] == "Tabular"].set_index("datos")["roc_auc"]
tabla_comb["delta_auc_vs_tabular"] = tabla_comb["roc_auc"] - tabla_comb["datos"].map(base)
tabla_comb[["datos", "modelo", "roc_auc", "pr_auc", "recall", "f1", "ks", "brier", "delta_auc_vs_tabular"]].round(4)

In [ ]:
plt.figure(figsize=(9, 4))
etiquetas = tabla_comb["datos"] + " | " + tabla_comb["modelo"]
plt.barh(etiquetas, tabla_comb["delta_auc_vs_tabular"],
         color=["tab:green" if v > 0 else "tab:gray" for v in tabla_comb["delta_auc_vs_tabular"]])
plt.axvline(0, color="k", lw=0.8)
plt.title("ΔAUC del texto sobre el modelo tabular"); plt.xlabel("ΔAUC (test)")
plt.tight_layout(); plt.show()

**Interpretación:** un ΔAUC positivo indica que el texto agrega información que las variables tabulares no capturan; un ΔAUC cercano a cero indica que el texto es redundante (por ejemplo, porque el cliente describe en palabras lo mismo que ya reflejan sus moras o su puntaje).

## 8. Guardar los resultados

In [ ]:
filas_json = [{"modelo": "TF-IDF + LR (texto, completo)", **met_tfidf}]
filas_json += [{"modelo": f"{r['modelo']} [{r['datos']}]", **{k: r[k] for k in met_tfidf}} for r in combinados if r["modelo"] != "Tabular"]
filas_json += [{"modelo": f"{n} [muestra]", **fila} for n, fila in tabla_txt.drop(index="TF-IDF + LR (muestra)").to_dict(orient="index").items()]
salida = {"fecha": pd.Timestamp.now().isoformat(timespec="seconds"), "n_muestra_llm": N_MUESTRA,
          "delta_auc": tabla_comb[["datos", "modelo", "delta_auc_vs_tabular"]].to_dict(orient="records"),
          "modelos": filas_json}
(MODELS_DIR / "resultados_nlp.json").write_text(json.dumps(salida, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("Guardado Back/models/resultados_nlp.json")

## 9. Resumen numérico

In [ ]:
print(f"TF-IDF + LR (solo texto, test completo): AUC = {met_tfidf['roc_auc']:.4f}")
print(tabla_txt["roc_auc"].round(4).to_string())
print(tabla_comb[["datos", "modelo", "roc_auc", "delta_auc_vs_tabular"]].round(4).to_string(index=False))

## Conclusiones

1. **Análisis de texto:** las solicitudes son textos cortos con vocabulario financiero; el *log-odds ratio* identifica términos ligados a atrasos, desempleo o consolidación de deudas (más frecuentes en impagos) y a cumplimiento y estabilidad (más frecuentes en buenos pagadores).
2. **TF-IDF + LR** es una línea base NLP fuerte, liviana y explicable: con solo el texto obtiene un AUC claramente superior a 0.5. Por eso es el modelo NLP que se sirve en la API (`/predict/text`).
3. **Sentimiento con BERT:** el puntaje de estrellas muestra una relación con el impago (tono más negativo, más riesgo), aunque débil, porque el modelo no fue entrenado para crédito.
4. **BERT vs GPT:** los embeddings de oración de BERT multilingüe superan a los de GPT-2 promediados, coherente con sus objetivos de entrenamiento (similitud semántica vs predicción de la siguiente palabra).
5. **Modelo combinado (IA3.E1):** agregar la señal del texto al modelo tabular produce un **ΔAUC pequeño** (ver la tabla del punto 7): el texto aporta poca información incremental sobre las variables de comportamiento crediticio, pero puede ser útil cuando faltan datos de buró.
6. **Costo:** BERT y GPT requieren descargas de ~1 GB y segundos por cada centenar de textos en CPU; por eso se cachean y no se usan en el modelo desplegado.